In [26]:
import pandas as pd
import numpy as np
import requests
from io import StringIO

# PM2.5 ----------------------------------------------------------

pm_base = "https://raw.githubusercontent.com/NazilaAbedini/ENGG680_2025_Fall/main/Final%20Project/Data/Raw_Data/PM2.5/"
pm_files = ["PM25_2020.csv", "PM25_2021.csv", "PM25_2022.csv", "PM25_2023.csv"]

def load_pm(url):
    text = requests.get(url).text
    df = pd.read_csv(StringIO(text), skiprows=7)
    date_col = [c for c in df.columns if "Date" in c][0]
    df["date"] = pd.to_datetime(df[date_col])
    hour_cols = [c for c in df.columns if c.startswith("H")]
    df["PM25"] = df[hour_cols].replace(-999, np.nan).mean(axis=1)
    return df[["date", "PM25"]]

pm_list = []
for fname in pm_files:
    url = pm_base + fname
    print("Loading PM:", fname)
    df = load_pm(url)
    pm_list.append(df)

df_pm25 = pd.concat(pm_list, ignore_index=True).sort_values("date")
print("PM2.5 final shape:", df_pm25.shape)



# Climate --------------------------------------------------------

cl_base = "https://raw.githubusercontent.com/NazilaAbedini/ENGG680_2025_Fall/main/Final%20Project/Data/Raw_Data/Climate/"
cl_files = [
    "en_climate_daily_AB_3031094_2020_P1D.csv",
    "en_climate_daily_AB_3031094_2021_P1D.csv",
    "en_climate_daily_AB_3031094_2022_P1D.csv",
    "en_climate_daily_AB_3031094_2023_P1D.csv"
]

def load_climate(url):
    text = requests.get(url).text
    df = pd.read_csv(StringIO(text))
    date_col = [c for c in df.columns if "Date" in c][0]
    df["date"] = pd.to_datetime(df[date_col])
    return df

cl_list = []
for fname in cl_files:
    url = cl_base + fname
    print("Loading Climate:", fname)
    df = load_climate(url)
    cl_list.append(df)

df_climate = pd.concat(cl_list, ignore_index=True).sort_values("date")
print("Climate final shape:", df_climate.shape)


Loading PM: PM25_2020.csv
Loading PM: PM25_2021.csv
Loading PM: PM25_2022.csv
Loading PM: PM25_2023.csv
PM2.5 final shape: (386424, 2)
Loading Climate: en_climate_daily_AB_3031094_2020_P1D.csv
Loading Climate: en_climate_daily_AB_3031094_2021_P1D.csv
Loading Climate: en_climate_daily_AB_3031094_2022_P1D.csv
Loading Climate: en_climate_daily_AB_3031094_2023_P1D.csv
Climate final shape: (1461, 32)


In [27]:
import pandas as pd
import numpy as np

pm_base = "https://raw.githubusercontent.com/NazilaAbedini/ENGG680_2025_Fall/main/Final%20Project/Data/Raw_Data/PM2.5/"
pm_files = [
    "PM25_2020.csv",
    "PM25_2021.csv",
    "PM25_2022.csv",
    "PM25_2023.csv",
]

pm_list = []

for fname in pm_files:
    url = pm_base + fname
    print("Loading:", fname)

    df = pd.read_csv(url, skiprows=7, encoding="utf-8-sig")

    date_col = [c for c in df.columns if "Date" in c][0]
    df["date"] = pd.to_datetime(df[date_col])

    hour_cols = [c for c in df.columns if c.startswith("H")]

    df["PM25"] = df[hour_cols].replace(-999, np.nan).mean(axis=1)

    pm_list.append(df[["date", "PM25"]])

pm_raw = pd.concat(pm_list, ignore_index=True)
pm_raw = pm_raw.sort_values("date")

pm_daily = pm_raw.groupby("date", as_index=False)["PM25"].mean()

pm_daily.head()



Loading: PM25_2020.csv
Loading: PM25_2021.csv
Loading: PM25_2022.csv
Loading: PM25_2023.csv


,date,PM25
0,2020-01-01,5.080895
1,2020-01-02,4.965239
2,2020-01-03,6.367912
3,2020-01-04,4.359544
4,2020-01-05,3.706763


In [28]:
import pandas as pd
import numpy as np
import requests
import io

base_url = "https://raw.githubusercontent.com/NazilaAbedini/ENGG680_2025_Fall/main/Final%20Project/Data/Raw_Data/Climate/"

weather_files = [
    "en_climate_daily_AB_3031094_2020_P1D.csv",
    "en_climate_daily_AB_3031094_2021_P1D.csv",
    "en_climate_daily_AB_3031094_2022_P1D.csv",
    "en_climate_daily_AB_3031094_2023_P1D.csv"
]

weather_list = []

for fname in weather_files:
    print("Loading climate:", fname)
    url = base_url + fname
    response = requests.get(url)
    df = pd.read_csv(io.BytesIO(response.content))
    df["date"] = pd.to_datetime(df["Date/Time"])
    df = df[["date", "Mean Temp (°C)", "Total Precip (mm)"]]
    weather_list.append(df)

weather_raw = pd.concat(weather_list, ignore_index=True)
weather_raw = weather_raw.sort_values("date")

weather_raw.head()


Loading climate: en_climate_daily_AB_3031094_2020_P1D.csv
Loading climate: en_climate_daily_AB_3031094_2021_P1D.csv
Loading climate: en_climate_daily_AB_3031094_2022_P1D.csv
Loading climate: en_climate_daily_AB_3031094_2023_P1D.csv


,date,Mean Temp (°C),Total Precip (mm)
0,2020-01-01,-1.6,0.0
1,2020-01-02,-4.2,1.4
2,2020-01-03,-0.4,0.0
3,2020-01-04,2.4,0.6
4,2020-01-05,-4.1,0.1


In [29]:
# 5 – Merge PM daily with Weather daily
df_merge = pd.merge(pm_daily, weather_raw, on="date", how="inner")

# 5.1 – Create lag and rolling features (same as original v2)
df_merge["PM25_lag1"] = df_merge["PM25"].shift(1)
df_merge["PM25_roll3"] = df_merge["PM25"].rolling(window=3).mean()

# 5.2 – Time-based features
df_merge["month"] = df_merge["date"].dt.month
df_merge["dow"] = df_merge["date"].dt.dayofweek

df_merge["season"] = df_merge["month"].apply(
    lambda m: "Winter" if m in [12, 1, 2]
    else "Spring" if m in [3, 4, 5]
    else "Summer" if m in [6, 7, 8]
    else "Fall"
)

df_merge.head()


,date,PM25,Mean Temp (°C),Total Precip (mm),PM25_lag1,PM25_roll3,month,dow,season
0,2020-01-01,5.080895,-1.6,0.0,NaN,NaN,1,2,Winter
1,2020-01-02,4.965239,-4.2,1.4,5.080895,NaN,1,3,Winter
2,2020-01-03,6.367912,-0.4,0.0,4.965239,5.471348,1,4,Winter
3,2020-01-04,4.359544,2.4,0.6,6.367912,5.230898,1,5,Winter
4,2020-01-05,3.706763,-4.1,0.1,4.359544,4.811406,1,6,Winter


In [30]:
# --- 6.0 Rename columns BEFORE the second merge (must match v3) ---
df_merge = df_merge.rename(columns={
    "Mean Temp (°C)": "Mean Temp",
    "Total Precip (mm)": "Total Preci"
})

weather_raw_renamed = weather_raw.rename(columns={
    "Mean Temp (°C)": "Mean Temp",
    "Total Precip (mm)": "Total Preci"
})

# --- 6.1 Second merge to create _x and _y columns ---
final_df = pd.merge(
    df_merge,
    weather_raw_renamed,
    on="date",
    how="inner",
    suffixes=("_x", "_y")
)

# --- 6.2 Add remaining features ---
final_df["year"] = final_df["date"].dt.year
final_df["dayofyear"] = final_df["date"].dt.dayofyear

# --- 6.3 Reorder columns EXACTLY like v3 ---
final_df = final_df[
    [
        "date",
        "PM25",
        "Mean Temp_x",
        "Total Preci_x",
        "PM25_lag1",
        "PM25_roll3",
        "month",
        "dow",
        "season",
        "Mean Temp_y",
        "Total Preci_y",
        "year",
        "dayofyear"
    ]
]

final_df.head()


,date,PM25,Mean Temp_x,Total Preci_x,PM25_lag1,PM25_roll3,month,dow,season,Mean Temp_y,Total Preci_y,year,dayofyear
0,2020-01-01,5.080895,-1.6,0.0,NaN,NaN,1,2,Winter,-1.6,0.0,2020,1
1,2020-01-02,4.965239,-4.2,1.4,5.080895,NaN,1,3,Winter,-4.2,1.4,2020,2
2,2020-01-03,6.367912,-0.4,0.0,4.965239,5.471348,1,4,Winter,-0.4,0.0,2020,3
3,2020-01-04,4.359544,2.4,0.6,6.367912,5.230898,1,5,Winter,2.4,0.6,2020,4
4,2020-01-05,3.706763,-4.1,0.1,4.359544,4.811406,1,6,Winter,-4.1,0.1,2020,5


In [31]:
# Fix column names to match the previous final dataset exactly
final_df = final_df.rename(columns={
    "Mean Temp_x": "Mean Temp (°C)_x",
    "Total Preci_x": "Total Precip (mm)_x",
    "Mean Temp_y": "Mean Temp (°C)_y",
    "Total Preci_y": "Total Precip (mm)_y"
})


In [32]:
output_path = "Calgary_PM25_Weather_Cleaned_v3.csv"

final_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Final dataset saved at:", output_path)


Final dataset saved at: Calgary_PM25_Weather_Cleaned_v3.csv
